In [ ]:
### Jupyter Notebook avec Commentaires Améliorés

#--------------------------------------------------------- Chargement des données--------------------------------------------------------------------------
# Importation des bibliothèques nécessaires pour la gestion des fichiers et des répertoires.
import os

# Changement du répertoire de travail vers le dossier parent contenant les données.
os.chdir("../data")

%pwd  # Affiche le répertoire de travail courant.

# Importation des outils pour charger et traiter les documents PDF.
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter


In [ ]:

#--------------------------------------------------------- Extraction des données--------------------------------------------------------------------------

# Définition d'une fonction pour charger les fichiers PDF d'un répertoire donné.
def load_pdf_file(data):
    loader = DirectoryLoader(data,  # Spécifie le répertoire contenant les fichiers PDF.
                             glob="*.pdf",  # Filtre pour ne traiter que les fichiers PDF.
                             loader_cls=PyPDFLoader)  # Utilise PyPDFLoader pour lire les fichiers PDF.

    documents = loader.load()  # Charge les documents et les retourne.
    return documents


# Chargement des données à partir du répertoire spécifié.
extracted_data = load_pdf_file(data="../data/")


# extracted_data - Vérification optionnelle des données extraites.

In [ ]:

#--------------------------------------------------------- Decoupage des données--------------------------------------------------------------------------

# Fonction pour diviser les données extraites en segments plus petits.
def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
    text_chunks = text_splitter.split_documents(extracted_data)  # Divise les documents en morceaux.
    return text_chunks


# Application de la fonction de découpage et impression du nombre de segments obtenus.
text_chunks = text_split(extracted_data)
print("Length of text Chunks", len(text_chunks))

# Installation et mise à jour des bibliothèques nécessaires.
!pip install -U langchain-huggingface
!pip uninstall sentence-transformers huggingface-hub langchain -y
!pip install sentence-transformers
!pip install huggingface-hub
!pip install langchain

# Importation du module pour utiliser des embeddings pré-entraînés.
from langchain_huggingface import HuggingFaceEmbeddings


In [ ]:

#--------------------------------------------------------- Téléchargement des embeddings --------------------------------------------------------------------------

# Fonction pour télécharger un modèle d'embedding depuis Hugging Face.
def download_hugging_face_embeddings():
    embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
    return embeddings


# Téléchargement des embeddings pour traiter les données textuelles.
embeddings = download_hugging_face_embeddings()

In [ ]:

#--------------------------------------------------------- Chargement des variables d'environnement --------------------------------------------------------------------------
from dotenv import load_dotenv
import os

# Chargement manuel des variables d'environnement depuis un fichier .env.
load_dotenv(override=True)

# Récupération des clés API stockées dans les variables d'environnement.
PINECONE_API_KEY = os.environ.get('PINECONE_API_KEY')
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY')

In [ ]:

#--------------------------------------------------------- Initialisation de Pinecone --------------------------------------------------------------------------
from pinecone.grpc import PineconeGRPC as Pinecone
from pinecone import ServerlessSpec

# Configuration de la connexion à Pinecone avec un délai d'attente spécifié.
pc = Pinecone(
    api_key=PINECONE_API_KEY,
    environment="us-east-1",  # Région pour l'hébergement des données.
    timeout=180  # Temps d'attente pour les connexions.
)

# Création d'un index vectoriel dans Pinecone pour stocker les embeddings.
index_name = "medicalbot"  # Nom de l'index utilisé.
pc.create_index(
    name=index_name,
    dimension=384,  # Dimension des vecteurs.
    metric="cosine",  # Métrique de similarité.
    spec=ServerlessSpec(cloud="aws", region="us-east-1")  # Déploiement serverless.
)

# Mise à jour des bibliothèques utilisées.
!pip install --upgrade langchain
!pip install --upgrade langchain-pinecone
!pip install --upgrade pinecone-client
!pip install --upgrade urllib3

# Ajout des clés API dans les variables d'environnement pour les rendre accessibles.
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [ ]:

#--------------------------------------------------------- Création d'index et intégration des embeddings --------------------------------------------------------------------------
from langchain_pinecone import PineconeVectorStore

# Création d'un index à partir des segments de texte et insertion des embeddings dans Pinecone.
docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks,
    index_name=index_name,
    embedding=embeddings,
)

# Chargement d'un index existant dans Pinecone pour la recherche.
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embeddings,
)
docsearch

In [ ]:

#--------------------------------------------------------- Création d'un récupérateur de documents --------------------------------------------------------------------------
retriever = docsearch.as_retriever(
    search_type="similarity",  # Recherche par similarité sémantique.
    search_kwargs={"k": 3}  # Retourne les 3 documents les plus similaires.
)

# Recherche des documents pertinents avec une requête textuelle.
retrieved_docs = retriever.invoke("What is Acne?")

# Affiche les documents récupérés.
retrieved_docs

In [ ]:

#--------------------------------------------------------- Intégration avec OpenAI --------------------------------------------------------------------------
from langchain_openai import OpenAI

# Initialisation du modèle OpenAI avec des paramètres personnalisés.
llm = OpenAI(
    temperature=0.4,  # Contrôle la créativité.
    max_tokens=500  # Limite de taille de réponse.
)

In [ ]:

#--------------------------------------------------------- Définition du prompt et création de chaînes --------------------------------------------------------------------------
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

# Définition du prompt pour guider les réponses générées.
system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise.\n\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

# Création des chaînes pour récupérer et traiter les informations pertinentes.
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [ ]:

#--------------------------------------------------------- Génération de réponse --------------------------------------------------------------------------
response = rag_chain.invoke({"input": "What is Acne?"})

# Affiche la réponse générée.
print(response["answer"])


